# 13 - Compute Human Evaluation

### Setup

In [ ]:
%pip install -q krippendorff huggingface_hub

import json
import os

import krippendorff
import numpy as np
import pandas as pd
from huggingface_hub import HfApi

## 1. Konfigurasi

In [ ]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_Faiss")
    except ImportError:
        token = os.environ.get("HF_Faiss")
        if token is None:
            raise RuntimeError("HF_TOKEN tidak ditemukan.")
        return token
 
HF_TOKEN = get_hf_token()
HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else "/workspace")
 
DIMENSIONS = ["fluency", "factual_correctness", "completeness"]
N_EVALUATORS = 2

## 2. Load hasil penilaian

In [ ]:
evaluator_dfs = []
for i in range(N_EVALUATORS):
    df = pd.read_csv(
        f"hf://datasets/{HF_REPO}/human_eval_evaluator{i+1}_filled.csv",
        storage_options={"token": HF_TOKEN},
    )
    evaluator_dfs.append(df)
 
df_key = pd.read_csv(
    f"hf://datasets/{HF_REPO}/human_eval_key.csv",
    storage_options={"token": HF_TOKEN},
)
 
n_rows = len(evaluator_dfs[0])
for i, df in enumerate(evaluator_dfs):
    assert len(df) == n_rows, f"Jumlah baris evaluator{i+1} beda dari evaluator1"
    assert (df["eval_id"].values == evaluator_dfs[0]["eval_id"].values).all(), f"Urutan eval_id evaluator{i+1} tidak sama dengan evaluator1"
assert len(df_key) == n_rows, "Jumlah baris key tidak sama dengan evaluator"

## 3. Hitung Krippendorff's Alpha per Dimensi

In [ ]:
def compute_alpha(list_of_score_lists):
    reliability_data = np.array(list_of_score_lists, dtype=float)
    return krippendorff.alpha(reliability_data=reliability_data, level_of_measurement="ordinal")
 
alpha_results = {}
for dim in DIMENSIONS:
    scores_per_evaluator = [df[dim].tolist() for df in evaluator_dfs]
    alpha = compute_alpha(scores_per_evaluator)
    alpha_results[dim] = alpha
    interpretation = "reliabel kuat" if alpha >= 0.80 else ("dapat diterima" if alpha >= 0.67 else "tidak memadai")
    print(f"{dim:22s}: alpha = {alpha:.4f} ({interpretation}) | n_evaluator={N_EVALUATORS}")

### buka key

In [ ]:
# ==== BUKA LABEL (merge dengan key) & RATA-RATA SKOR ANTAR SEMUA EVALUATOR ====
 
df_combined = evaluator_dfs[0][["eval_id"]].merge(df_key, on="eval_id", how="left")
for dim in DIMENSIONS:
    stacked = np.stack([df[dim].values for df in evaluator_dfs], axis=0)
    df_combined[dim] = stacked.mean(axis=0)
 
summary = df_combined.groupby(["_model_key", "_condition"])[DIMENSIONS].mean()
summary

## 4. Simpan Hasil

In [ ]:
# ==== SIMPAN & PUSH HASIL AKHIR KE HF HUB ====
 
result_summary = {
    "n_evaluators": N_EVALUATORS,
    "krippendorff_alpha": alpha_results,
    "likert_mean_by_model_condition": {f"{k[0]}_{k[1]}": v for k, v in summary.to_dict(orient="index").items()},
}
 
result_path = f"{OUTPUT_DIR}/human_eval_result.json"
with open(result_path, "w") as f:
    json.dump(result_summary, f, indent=2)
 
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=result_path,
    path_in_repo="human_eval_result.json",
    repo_id=HF_REPO,
    repo_type="dataset",
)
print(f"Ter-push ke HF Hub: {HF_REPO}/human_eval_result.json")